In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE


In [ ]:
df = pd.read_csv('mydata.csv')
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label'],
    test_size=0.2, stratify=df['label'], random_state=42
)

In [ ]:
# наивный байес
# метод опорных векторов
svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50_000,
        ngram_range=(1,2),
        stop_words='english'
    )),
    ('svm', LinearSVC(
        C=1.0,
        class_weight='balanced',
        max_iter=10_000,
        random_state=42
    ))
])

In [ ]:
svm_pipeline.fit(X_train, y_train)

In [ ]:
y_pred = svm_pipeline.predict(X_test)
print("SVM Accuracy: ", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=4))

In [ ]:
# Предполагается, что svm_pipeline уже обучен, а y_pred получены:
# y_pred = svm_pipeline.predict(X_test)

labels = svm_pipeline.classes_  # порядок классов
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(8,6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=labels,
    yticklabels=labels
)
plt.xlabel('Предсказанный класс')
plt.ylabel('Истинный класс')
plt.title('Матрица ошибок (SVM)')
plt.tight_layout()
plt.show()


In [ ]:
# Проверяем, что это действительно бинарная задача
if len(svm_pipeline.classes_) != 2:
    raise ValueError(f"ROC‐кривую можно строить только для двух классов, а у вас {len(svm_pipeline.classes_)}")

# Получаем "оценки" (scores) методом decision_function
y_scores = svm_pipeline.decision_function(X_test)  # shape = (n_samples,)

# Мапим y_test в 0/1, где classes_[1] считаем «положительным»
mapping = {svm_pipeline.classes_[0]: 0, svm_pipeline.classes_[1]: 1}
y_test_bin = y_test.map(mapping)

# ROC‐метрики
fpr, tpr, thresholds = roc_curve(y_test_bin, y_scores)
roc_auc = auc(fpr, tpr)

In [ ]:
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.4f}')
plt.plot([0,1], [0,1], 'k--', linewidth=0.8)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC‐кривая (SVM)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# 1) Берём тот же TfidfVectorizer из пайплайна
tfidf = svm_pipeline.named_steps['tfidf']
X_all_tfidf = tfidf.transform(df['clean_text'])  # можно взять весь df или только X_test/X_train

# 2) Снижаем размерность через TruncatedSVD до 50 компонент
svd = TruncatedSVD(n_components=50, random_state=42)
X_svd = svd.fit_transform(X_all_tfidf)

# 3) Применяем t-SNE (2D) к выходу SVD
tsne = TSNE(
    n_components=2,
    random_state=42,
    init='pca',
    learning_rate='auto'
)
X_tsne = tsne.fit_transform(X_svd)


In [ ]:
# 4) Строим scatter plot, раскрашивая по меткам
plt.figure(figsize=(8,6))
for label in df['label'].unique():
    idx = df['label'] == label
    plt.scatter(
        X_tsne[idx, 0],
        X_tsne[idx, 1],
        label=label,
        alpha=0.6,
        s=20
    )

plt.xlabel('t-SNE компонента 1')
plt.ylabel('t-SNE компонента 2')
plt.title('t-SNE проекция TF-IDF признаков (SVM)')
plt.legend(markerscale=2, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_curve, auc
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Загрузим данные и заменим метки 0→"human", 1→"AI"
df = pd.read_csv('mydata.csv')
label_mapping = {0: 'human', 1: 'AI'}
df['label'] = df['label'].map(label_mapping)

# 2. Разбиваем на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'],
    df['label'],
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

# 3. Строим пайплайн TF-IDF + LinearSVC
svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50_000,
        ngram_range=(1, 2),
        stop_words='english'
    )),
    ('svm', LinearSVC(
        C=1.0,
        class_weight='balanced',
        max_iter=10_000,
        random_state=42
    ))
])

# 4. Обучаем SVM
svm_pipeline.fit(X_train, y_train)

# 5. Предсказываем и оцениваем качество
y_pred = svm_pipeline.predict(X_test)
print("SVM Accuracy:", accuracy_score(y_test, y_pred))
print(
    classification_report(
        y_test,
        y_pred,
        labels=['human', 'AI'],       # явно указываем порядок
        target_names=['human', 'AI'], # имена классов
        digits=4
    )
)

# 6. Матрица ошибок (confusion matrix) с надписями «human» и «AI»
cm_labels = ['human', 'AI']
cm = confusion_matrix(y_test, y_pred, labels=cm_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=cm_labels,
    yticklabels=cm_labels
)
plt.xlabel('Предсказанный класс')
plt.ylabel('Истинный класс')
plt.title('Матрица ошибок (SVM)')
plt.tight_layout()
plt.show()

In [ ]:
# 7. Проверяем, что это бинарная классификация
if len(svm_pipeline.classes_) != 2:
    raise ValueError(
        f"ROC‐кривую можно строить только для двух классов, а у вас {len(svm_pipeline.classes_)}"
    )

# 8. Получаем decision scores и мапим y_test в 0/1 (0="human", 1="AI")
y_scores = svm_pipeline.decision_function(X_test)
mapping = {svm_pipeline.classes_[0]: 0, svm_pipeline.classes_[1]: 1}
y_test_bin = y_test.map(mapping)

# 9. Считаем ROC‐метрики и рисуем ROC‐кривую
fpr, tpr, thresholds = roc_curve(y_test_bin, y_scores)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.4f}')
plt.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC‐кривая (SVM)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# 10. Трансформация всех текстов в TF-IDF (используем tfidf из пайплайна)
tfidf = svm_pipeline.named_steps['tfidf']
X_all_tfidf = tfidf.transform(df['clean_text'])

# 11. Снижаем размерность через TruncatedSVD до 50 компонент
svd = TruncatedSVD(n_components=50, random_state=42)
X_svd = svd.fit_transform(X_all_tfidf)

# 12. Применяем t-SNE (2D) к выходу SVD
tsne = TSNE(
    n_components=2,
    random_state=42,
    init='pca',
    learning_rate='auto'
)
X_tsne = tsne.fit_transform(X_svd)

# 13. Строим scatter plot, раскрашивая по меткам «human» и «AI»
plt.figure(figsize=(8, 6))
for label in ['human', 'AI']:
    idx = (df['label'] == label)
    plt.scatter(
        X_tsne[idx, 0],
        X_tsne[idx, 1],
        label=label,
        alpha=0.6,
        s=20
    )

plt.xlabel('t-SNE компонента 1')
plt.ylabel('t-SNE компонента 2')
plt.title('t-SNE проекция TF-IDF признаков (SVM)')
plt.legend(markerscale=2, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()
